# First Training file combining text and visual features

In [7]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3050 Laptop GPU


In [1]:
import pandas as pd

In [8]:
train = pd.read_csv('train_processed.csv')

In [9]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75000 entries, 0 to 74999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   sample_id    75000 non-null  int64  
 1   value        75000 non-null  float64
 2   model_input  75000 non-null  object 
 3   log_price    75000 non-null  float64
 4   image_name   75000 non-null  object 
dtypes: float64(2), int64(1), object(2)
memory usage: 2.9+ MB


In [10]:
train.head()

,sample_id,value,model_input,log_price,image_name
0,33127,72.00,"La Victoria Green Taco Sauce Mild, 12 Ounce (P...",1.773256,51mo8htwTHL.jpg
1,198967,32.00,"Salerno Cookies, The Original Butter Cookies, ...",2.647592,71YtriIHAAL.jpg
2,261251,11.40,"Bear Creek Hearty Soup Bowl, Creamy Chicken wi...",1.088562,51+PFEe-w-L.jpg
3,55858,11.25,Judee’s Blue Cheese Powder 11.25 oz - Gluten-F...,3.444895,41mu0HAToDL.jpg
4,292686,12.00,"kedem Sherry Cooking Wine, 12.7 Ounce - 12 per...",4.211979,41sA037+QvL.jpg


In [11]:
train.isnull().sum()

sample_id      0
value          0
model_input    0
log_price      0
image_name     0
dtype: int64

In [12]:
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

/home/gagan/Desktop/side-projects/amazon-mlchallenge/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
from sklearn.model_selection import train_test_split
X = train[['sample_id','value', 'model_input']]
y = train['log_price']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [14]:
import re
import string

def clean_text(text):
    if pd.isnull(text):
        return ""
    # Remove emojis
    text = re.sub(r'[\U00010000-\U0010ffff]', '', text)
    # Remove HTML tags
    text = re.sub(r'<.*?>', ' ', text)
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', ' ', text)
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Lowercase
    text = text.lower()
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Clean train/val/test model_input
X_train['model_input'] = X_train['model_input'].apply(clean_text)
X_val['model_input'] = X_val['model_input'].apply(clean_text)


In [15]:
# see first model_input
print(X_train['model_input'].iloc[0])
print(X_train['sample_id'].iloc[0])

tiesta tea fruity loose leaf dry flight mother’s day tea gift set sampler caffeinefree hot iced ready assorted fruit blends with mango peach orange more 8 resealable sample pouches ounce experience the vibrant world of tiesta tea’s fruity sampler dry flight tea set this set invites you to explore a variety of refreshing loose leaf tea blends each inspired by the essence of real fruit with 8 unique caffeinefree teas including flavors like maui mango blueberry wild child and strawberry lemonade there’s a tea to match every taste each blend brews a rich aromatic cup allowing for a more flavorful experience compared to traditional bagged teas conveniently packaged in resealable pouches these teas stay fresh and are perfect for brewing 610 cups per sample enjoy your tea moment with tiesta tea explore a world of flavors enjoy a variety of flavors with tiesta tea’s fruity sampler dry flight tea set this sampler set features 8 unique loose leaf tea blends each offering a distinct fruitinspired

In [26]:
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import numpy as np
import os
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(
    'google-bert/bert-base-uncased',
    use_fast=True,
    local_files_only=False
)
bert_model = AutoModel.from_pretrained("google-bert/bert-base-uncased")
bert_model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model.to(device)

def get_bert_embeddings(texts, tokenizer, model, batch_size=32, max_length=128):
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        input_ids = encoded['input_ids'].to(device)
        attention_mask = encoded['attention_mask'].to(device)
        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            # Use [CLS] token embedding as sentence embedding
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls_embeddings)
    return np.vstack(embeddings)

# Example usage for train/val splits
X_train_emb = get_bert_embeddings(X_train['model_input'].tolist(), tokenizer, bert_model)
X_val_emb = get_bert_embeddings(X_val['model_input'].tolist(), tokenizer, bert_model)

100%|██████████| 463/463 [00:54<00:00,  8.48it/s]


In [27]:
scaler = StandardScaler()
X_train_value = scaler.fit_transform(X_train[['value']])
X_val_value = scaler.transform(X_val[['value']])

In [28]:
X_train_combined = np.hstack([X_train_value, X_train_emb])
X_val_combined = np.hstack([X_val_value, X_val_emb])

In [29]:
class PriceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(np.array(y), dtype=torch.float32)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = PriceDataset(X_train_combined, y_train.values)
val_ds = PriceDataset(X_val_combined, y_val.values)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=128)

# 6. Improved Neural Network
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLP(X_train_combined.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# 7. Training loop with early stopping
best_loss = float('inf')
patience = 5
counter = 0
epochs = 50

for epoch in range(epochs):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb)
        loss = loss_fn(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    # Validation
    model.eval()
    val_losses = []
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            val_loss = loss_fn(pred, yb)
            val_losses.append(val_loss.item())
    avg_val_loss = np.mean(val_losses)
    print(f"Epoch {epoch+1}, Val Loss: {avg_val_loss:.4f}")
    # Early stopping
    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        counter = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping!")
            break

# 8. Load best model and evaluate SMAPE
model.load_state_dict(torch.load("best_model.pt"))
model.eval()
preds = []
with torch.no_grad():
    for xb, _ in val_dl:
        xb = xb.to(device)
        pred = model(xb).cpu().numpy()
        preds.append(pred)
y_pred = np.concatenate(preds)

def smape(y_true, y_pred):
    y_true = np.expm1(y_true)
    y_pred = np.expm1(y_pred)
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8))

print(f"Validation SMAPE: {smape(y_val.values, y_pred):.2f}%")

Epoch 1, Val Loss: 0.6789
Epoch 2, Val Loss: 0.6292
Epoch 3, Val Loss: 0.6112
Epoch 4, Val Loss: 0.6051
Epoch 5, Val Loss: 0.6048
Epoch 6, Val Loss: 0.5965
Epoch 7, Val Loss: 0.5941
Epoch 8, Val Loss: 0.5795
Epoch 9, Val Loss: 0.5902
Epoch 10, Val Loss: 0.6095
Epoch 11, Val Loss: 0.6467
Epoch 12, Val Loss: 0.6325
Epoch 13, Val Loss: 0.5756
Epoch 14, Val Loss: 0.5657
Epoch 15, Val Loss: 0.5633
Epoch 16, Val Loss: 0.7564
Epoch 17, Val Loss: 0.5542
Epoch 18, Val Loss: 0.5761
Epoch 19, Val Loss: 0.5603
Epoch 20, Val Loss: 0.5618
Epoch 21, Val Loss: 0.5594
Epoch 22, Val Loss: 0.5648
Early stopping!
Validation SMAPE: 57.52%


In [17]:
test = pd.read_csv('test_processed.csv')

In [18]:
test.head()

,sample_id,value,model_input,image_name
0,100179,10.5,Rani 14-Spice Eshamaya's Mango Chutney (Indian...,71hoAn78AWL.jpg
1,245611,2.0,Natural MILK TEA Flavoring extract by HALO PAN...,61ex8NHCIjL.jpg
2,146263,32.0,Honey Filled Hard Candy - Bulk Pack 2 Pounds -...,61KCM61J8eL.jpg
3,95658,2.0,Vlasic Snack'mm's Kosher Dill 16 Oz (Pack of 2...,51Ex6uOH7yL.jpg
4,36806,32.0,"McCormick Culinary Vanilla Extract, 32 fl oz -...",71QYlrOMoSL.jpg


In [31]:
test.isnull().sum()

sample_id      0
value          0
model_input    1
dtype: int64

In [32]:
test['model_input'] = test['model_input'].fillna('')

In [33]:
test_emb = get_bert_embeddings(test['model_input'].tolist(), tokenizer, bert_model)

100%|██████████| 2344/2344 [04:35<00:00,  8.50it/s]


In [34]:
test_value = scaler.transform(test[['value']])

In [35]:
test_combined = np.hstack([test_value, test_emb])

In [36]:
test_tensor = torch.tensor(test_combined, dtype=torch.float32).to(device)
model.eval()
with torch.no_grad():
    pred_log_price = model(test_tensor).cpu().numpy()
pred_price = np.expm1(pred_log_price).clip(0)

In [37]:
submission = pd.DataFrame({
    'sample_id': test['sample_id'],
    'price': pred_price
})

In [38]:
submission.to_csv('submission.csv', index=False)
print("Submission file saved as submission.csv")

Submission file saved as submission.csv
